Installs all required Python dependencies for data analysis, geospatial processing, and PostgreSQL database access.

In [ ]:
from IPython.display import clear_output
clear_output(wait=True)
%pip install -q pandas sqlalchemy geopandas shapely psycopg2 geoalchemy2 seaborn scikit-learn python-dotenv jupysql plotly requests

Loads the SQL extension to enable running SQL queries directly within Jupyter notebook cells.

In [ ]:
%load_ext sql

Imports the necessary Python libraries and defines the path to directory where the data files are stored.

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import geopandas as gpd
import numpy as np
from pathlib import Path
import os
from dotenv import load_dotenv
import plotly.express as px


load_dotenv("project.env")

db_url = os.getenv("DATABASE_URL")

engine = create_engine(db_url)
data_dir = Path("data")

In [ ]:
%sql engine

Loads the SA2 boundary shapefile into a GeoDataFrame, filters for the Greater Sydney region, standardises column names, and uploads the result to the PostgreSQL database as the `sa2_boundaries` table.

In [ ]:
sa2_folder_path = data_dir / "SA2_2021_AUST_SHP_GDA2020"
sa2_shapefile = f"{sa2_folder_path}/SA2_2021_AUST_GDA2020.shp"

gdf_sa2 = gpd.read_file(sa2_shapefile)
gdf_sa2.columns = gdf_sa2.columns.str.replace(r'21$', '', regex=True).str.lower()
gdf_sa2 = gdf_sa2[gdf_sa2['gcc_name'] == 'Greater Sydney']
gdf_sa2.to_postgis("sa2_boundaries", engine, if_exists="replace", index=False)

Extracting co-ordinates from suburb shapefile. Filtering to the NSW and cleaning the columns.

In [ ]:
sal_shapefile = data_dir / "SAL_2021_AUST_GDA2020_SHP/SAL_2021_AUST_GDA2020.shp"
gdf_sal = gpd.read_file(sal_shapefile)

gdf_sal.columns = gdf_sal.columns.str.replace(r'21$', '', regex=True).str.lower()
gdf_sal = gdf_sal[gdf_sal['ste_name'] == 'New South Wales']
gdf_sal.to_postgis("suburbs", engine, if_exists="replace", index=False)


Loads the business data from business.csv into the pandas DataFrames, removes duplicate data, and uploads these DataFrames to the PostgreSQL database, creating the `business` table.

In [ ]:
df_business = pd.read_csv(data_dir / "Businesses.csv")
df_business.drop_duplicates(inplace=True)
df_business.to_sql("businesses", engine, if_exists="replace", index=False)

Loads the monthly crime data, filtering to the crimes which makes the locality less safe.

Filtering for the 2019 and 2020 crime data and summing them up grouped by the suburb.

In [ ]:

crime_csv = data_dir / "SuburbData25Q3.csv"
df_crime = pd.read_csv(crime_csv)
df_crime.columns = df_crime.columns.str.strip().str.lower().str.replace(' ', '_')

liveability_crimes = [
    # Violent & Personal Crimes
    'Murder *', 'Attempted murder', 'Murder accessory, conspiracy', 'Manslaughter *', 
    'Domestic violence related assault', 'Non-domestic violence related assault', 
    'Assault Police', 'Sexual assault', 'Sexual touching, sexual act and other sexual offences', 
    'Abduction and kidnapping', 'Robbery without a weapon', 'Robbery with a firearm', 
    'Robbery with a weapon not a firearm', 'Blackmail and extortion', 'Coercive Control', 
    'Intimidation, stalking and harassment', 'Other offences against the person',
    'Prohibited and regulated weapons offences',
    
    # Property, Theft & Damage
    'Break and enter dwelling', 'Break and enter non-dwelling', 'Receiving or handling stolen goods', 
    'Motor vehicle theft', 'Steal from motor vehicle', 'Steal from retail store', 
    'Steal from dwelling', 'Steal from person', 'Arson', 'Malicious damage to property', 
    'Trespass', 'Criminal intent',
    
    # Severe Drug Crime (Manufacturing/Trafficking only, NO possession)
    'Dealing, trafficking in cocaine', 'Dealing, trafficking in narcotics', 
    'Dealing, trafficking in cannabis', 'Dealing, trafficking in amphetamines', 
    'Dealing, trafficking in ecstasy', 'Dealing, trafficking in other drugs', 
    'Cultivating cannabis', 'Manufacture drug', 'Importing drugs'
]

df_filtered = df_crime[df_crime['subcategory'].isin(liveability_crimes)].copy()

crime_cols = df_filtered.filter(regex='2019|2020').columns
df_filtered['total_crimes'] = df_filtered[crime_cols].sum(axis=1)
df_crime_clean = df_filtered.groupby('suburb')['total_crimes'].sum().reset_index()


Uploading the crime data to the PostgreSQL database as the `crime_raw` table.

In [ ]:
df_crime_clean.to_sql("crime_raw", engine, if_exists="replace", index=False)

Loads the stops data from stops.txt into a pandas DataFrame, processes the coordinates into a GeoDataFrame, and uploads the result to the PostgreSQL database as the `stops` table.

In [ ]:
df_stops = pd.read_csv(data_dir / "Stops.txt")
df_stops['stop_code'] = df_stops['stop_code'].astype('Int64')
df_stops['wheelchair_boarding'] = df_stops['wheelchair_boarding'].map({1: True, 0: False})

gdf_stops = gpd.GeoDataFrame(df_stops, 
                            geometry=gpd.points_from_xy(df_stops.stop_lon, df_stops.stop_lat),
                            crs="EPSG:4326")

gdf_stops.to_postgis("stops", engine, if_exists="replace", index=False)

Loads the primary and secondary school catchment shapefiles into GeoDataFrames, cleans and processes the data, and uploads them to the PostgreSQL database as the `catchments_primary` and `catchments_secondary` tables. 

In [ ]:
primary_path = data_dir / "catchments/catchments_primary.shp"
secondary_path = data_dir / "catchments/catchments_secondary.shp"

gdf_primary = gpd.read_file(primary_path)
gdf_secondary = gpd.read_file(secondary_path)


gdf_primary.columns = gdf_primary.columns.str.lower()
gdf_secondary.columns = gdf_secondary.columns.str.lower()

gdf_primary['add_date'] = pd.to_datetime(gdf_primary['add_date'], format='%Y%m%d').dt.date
gdf_secondary['add_date'] = pd.to_datetime(gdf_secondary['add_date'], format='%Y%m%d').dt.date

columns_to_convert = ['kindergart', 'year1', 'year2', 'year3', 'year4',
                      'year5', 'year6', 'year7', 'year8', 'year9', 'year10',
                      'year11', 'year12']

gdf_primary[columns_to_convert] = gdf_primary[columns_to_convert] == 'Y'
gdf_secondary[columns_to_convert] = gdf_secondary[columns_to_convert] == 'Y'

gdf_primary.to_postgis("catchments_primary", engine, if_exists="replace", index=False)
gdf_secondary.to_postgis("catchments_secondary", engine, if_exists="replace", index=False)

Loads the population data from population.csv into the pandas DataFrames, and then uploads these DataFrames to the PostgreSQL database, creating the `population` table.

In [ ]:
df_population = pd.read_csv(data_dir / "Population.csv")
df_population.to_sql("population", engine, if_exists="replace", index=False)

Retrieves Points of Interest (POIs) from the NSW government API for each SA2 region by querying the API with the bounding box of each SA2 geometry. It collects the POI attributes and coordinates, creates a pandas DataFrame, then converts it into a GeoDataFrame. Finally, it uploads the POIs to the PostgreSQL database as the `pois` table.

In [ ]:
import concurrent.futures
import requests
import json


def pois_in_bbox(minx, miny, maxx, maxy, offset=0):
    base_url = 'https://maps.six.nsw.gov.au/arcgis/rest/services/public/NSW_POI/MapServer/0/query'
    
    geometry = json.dumps({
        "xmin": minx,
        "xmax": maxx,
        "ymin": miny,
        "ymax": maxy,
        "spatialReference": {"wkid": 4326}
    })
    
    params = {
        'geometry': geometry,
        'geometryType': 'esriGeometryEnvelope',
        'spatialRel': 'esriSpatialRelIntersects',
        'outFields': '*',
        'returnGeometry': 'true',
        'inSR': '4326',
        'outSR': '4326',
        'f': 'json',
        'resultOffset': offset,
        'resultRecordCount': 1000
    }
    
    try:
        response = requests.get(base_url, params=params, timeout=30)
        if response.status_code == 200:
            data = response.json()
            features = data.get('features', [])
            if data.get('exceededTransferLimit', False):
                print(f"  Paginating from offset {offset + 1000}...")
                features += pois_in_bbox(minx, miny, maxx, maxy, offset + 1000)
            return features
        else:
            return []
    except Exception as e:
        print(f"  Request failed at offset {offset}: {e}")
        return []


def process_single_region(row_data):
    """Worker function for parallel processing"""
    index, row = row_data 
    
    bounds = row.geometry.bounds
    
    pois = pois_in_bbox(*bounds) 
    
    local_list = []
    for poi in pois:
        attributes = poi.get('attributes', {})
        geometry = poi.get('geometry', {})
        
        if attributes and geometry:
            local_list.append({
                'poi_name' : attributes.get('poiname'),
                'poi_group' : attributes.get('poigroup'),
                'poi_type' : attributes.get('poitype'),
                'x': geometry.get('x'),
                'y': geometry.get('y')
            })
    return local_list



all_pois = []
rows_to_process = list(gdf_sa2.iterrows())
total = len(rows_to_process)

print(f"Starting parallel download for {total} regions...")

with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:

    futures = {executor.submit(process_single_region, item): item for item in rows_to_process}
    
    completed_count = 0
    

    for future in concurrent.futures.as_completed(futures):
        try:
            result = future.result()
            all_pois.extend(result)
            
            completed_count += 1
            if completed_count % 10 == 0:
                print(f"Completed {completed_count}/{total} ({(completed_count/total)*100:.1f}%)")
                
        except Exception as e:
            print(f"Row failed: {e}")


if all_pois:
    df_pois = pd.DataFrame(all_pois)
    df_pois = df_pois.dropna(subset=['x', 'y'])
    df_pois = df_pois.drop_duplicates(subset=['poi_name', 'x', 'y'])
    
    gdf_pois = gpd.GeoDataFrame(df_pois,
                                geometry = gpd.points_from_xy(df_pois.x,df_pois.y),
                                crs = 'EPSG:4326')
    
    gdf_pois.to_postgis('pois', engine, if_exists='replace', index=False)
    print(f"Success! Uploaded {len(gdf_pois)} POIs to PostGIS.")
else:
    print("No POIs found.")

Adds a new column `sa2_name` of type TEXT to the `pois` table in the PostgreSQL database to store the SA2 region name for each point of interest.

In [ ]:
%%sql
ALTER TABLE pois 
ADD COLUMN IF NOT EXISTS sa2_name TEXT,
ADD COLUMN IF NOT EXISTS sa2_code TEXT;

In [ ]:
%%sql
CREATE INDEX IF NOT EXISTS idx_pois_group ON public.pois(poi_group);
CREATE INDEX IF NOT EXISTS idx_pois_type ON public.pois(poi_type);

Updates the `sa2_name` column in the `pois` table by assigning each point of interest the name of the SA2 region it falls within, using a spatial join with the `sa2_boundaries` table. The geometries are transformed to the same spatial reference system (SRID 7844) to ensure accurate matching.

In [ ]:
%%sql

UPDATE pois
SET 
    sa2_name = sa2.sa2_name,
    sa2_code = sa2.sa2_code
FROM sa2_boundaries sa2
WHERE ST_Within(ST_Transform(pois.geometry, 7844), sa2.geometry);

Creates the `selected_businesses` table by filtering the businesses table for specific industries: 'Public Administration and Safety', 'Accommodation and Food', 'Education and Training', 'Retail Trade', 'Health Care and Social Assistance', and 'Art and Recreation. It then groups this filtered data by `sa2_code` and sums the total_businesses for each area.  This process generates a table showing the count of these selected business types per `sa2_code`.  Essentially, the code provides a summary of industry concentration within each `sa2_code`.

In [ ]:
%%sql
DROP TABLE IF EXISTS selected_businesses;

CREATE TABLE selected_businesses AS
SELECT 
    sa2_code,
    SUM(total_businesses) AS business_count
FROM businesses
WHERE industry_name IN (
    'Public Administration and Safety',
    'Accommodation and Food Services',
    'Education and Training',
    'Retail Trade',
    'Health Care and Social Assistance',
    'Art and Recreation Services'
)
GROUP BY sa2_code;


Creates a table `sa2_stop_counts` by counting the number of stops within each SA2 region. It joins the sa2_boundaries table with the stops table, using the ST_Contains function to determine which stops fall within each SA2 region's geometry. The ST_Transform function ensures both geometries are in the same spatial reference system (SRID 7844) for accurate comparison. Finally, it groups the results by `sa2_code` and `sa2_name` to provide a count of stops for each SA2 area, storing this count in the `stop_count` column.

In [ ]:
%%sql
DROP TABLE IF EXISTS sa2_stop_counts;

CREATE TABLE sa2_stop_counts AS
SELECT
    s.sa2_code,
    s.sa2_name,
    COUNT(pt.stop_id) AS stop_count
FROM sa2_boundaries s
JOIN stops pt
    ON ST_Contains(s.geometry, ST_Transform(pt.geometry, 7844))
GROUP BY s.sa2_code, s.sa2_name;

Creates a new table `schools_sa2` by combining all records from the `catchments_primary` and `catchments_secondary` tables, resulting in a single table containing both primary and secondary school catchment data for further analysis.

In [ ]:
%%sql
DROP TABLE IF EXISTS schools_sa2;

CREATE TABLE schools_sa2 AS
SELECT *
FROM catchments_primary
UNION ALL
SELECT *
FROM catchments_secondary;

Creates a new table `schools`.  It starts by selecting all columns from the schools_sa2 table (aliased as s).  It then performs a spatial join with the `sa2_boundaries` table (aliased as sb) using the ST_Intersects function.  The ST_Transform function is used to ensure the geometry columns from both tables are in the same spatial reference system (SRID 7844) before performing the intersection.  Finally, it adds the `sa2_code` and `sa2_name` from the `sa2_boundaries` table to `schools`.  In essence, this code enriches the `schools_sa2` table with SA2 information by matching schools to the SA2 regions they fall within.

Additionally creates `schools_per_sa2`.  It selects the `sa2_code` and `sa2_name` from the schools table (created by the first code block).  It then counts the number of schools within each SA2 region by counting the occurrences of `use_desc` and groups the results by `sa2_code` and `sa2_name`.  The final table, `schools_per_sa2`, contains the columns `sa2_code`, `sa2_name`, and `school_count`, representing the number of schools in each SA2 region.

In [ ]:
%%sql
DROP TABLE IF EXISTS schools;
DROP TABLE IF EXISTS schools_per_sa2;

CREATE TABLE schools AS
SELECT s.*, sb.sa2_code, sb.sa2_name
FROM schools_sa2 s
JOIN sa2_boundaries sb
ON ST_Intersects(ST_Transform(s.geometry, 7844), sb.geometry);

CREATE TABLE schools_per_sa2 AS
SELECT 
    sa2_code, 
    sa2_name, 
    COUNT(use_desc) AS school_count
FROM schools
GROUP BY sa2_code, sa2_name;

Creates a table `young_population`.  It calculates the total number of young people (aged 0-19) for each SA2 region.  It does this by summing the population counts from the population table for the age groups "0_4_people", "5_9_people", "10_14_people", and "15_19_people".  The results are grouped by `sa2_code`, with the sum representing the young population for that SA2 region.

In [ ]:
%%sql
DROP TABLE IF EXISTS young_population;

CREATE TABLE young_population AS
SELECT 
    sa2_code,
    SUM("0-4_people" + "5-9_people" + "10-14_people" + "15-19_people") AS young_population
FROM population
GROUP BY sa2_code;

Creates a table `selected_pois`.  It counts specific Points of Interest (POIs) within each SA2 region.  The code joins the pois table with the sa2_boundaries table using a spatial function (ST_Contains) to determine which SA2 region each POI is located in.  Before performing this check, it transforms the POI geometries to SRID 7844 to ensure coordinate system compatibility.  It filters the POIs to include those with `poi_group` values of 1, 2, 3, or 4, or those with a `poi_group` of 6 and a `poi_type` of 'Beach'.  Finally, it groups the POIs by `sa2_name` and counts the number of POIs in each group, storing the counts in the `poi_count` column.

We chose to include the `poi_type` 'Beach' from group 6 as we believed it is an essential part of recreation for many Australian families. 

In [ ]:
%%sql
DROP TABLE IF EXISTS selected_pois;

CREATE TABLE selected_pois AS
SELECT 
    sb.sa2_name,
    sb.sa2_code,
    COUNT(*) AS poi_count
FROM pois p
JOIN sa2_boundaries sb
    ON ST_Contains(sb.geometry, ST_Transform(p.geometry, 7844))
WHERE 
    poi_group IN (1, 2, 3, 4)
    OR (poi_group = 6 AND poi_type = 'Beach')
GROUP BY sb.sa2_name, sb.sa2_code;

Disables JupySQL's named parameter detection, which incorrectly flags PostgreSQL's `::geography` cast syntax as Python template variables.

In [ ]:
%config SqlMagic.named_parameters="disabled"


### Area-Weighted Spatial Allocation (Crime Data)

This pipeline step solves the "Spatial Mismatch" problem between NSW Police reporting boundaries (Suburbs) and ABS Census boundaries (SA2s). Because a single suburb can sprawl across multiple SA2s, performing a standard 1:1 relational join would result in massive double-counting of crime statistics.

To ensure mathematical accuracy, we use **Area-Weighted Spatial Interpolation**:

* **True Spherical Mathematics:** We cast the raw geometries to PostGIS's native `geography` type. This forces the area calculations (`ST_Area`) to be executed over the true spherical shape of the Earth in exact square meters, neutralizing the distortion caused by GDA2020 geographic coordinate degrees.

* **Dynamic Overlap Ratios:** We calculate exactly what percentage of a suburb's landmass falls inside an intersecting SA2 polygon (`ST_Intersection`).

* **Proportional Distribution:** The raw crime counts are multiplied by this `overlap_ratio`. If 65% of a high-crime suburb's physical area falls into SA2 'A', and 35% falls into SA2 'B', the crime count is split 65/35 between them.

* **String Cleansing:** We utilize `SPLIT_PART` on the fly to strip trailing state identifiers (e.g., removing " (NSW)") from the spatial boundary names to cleanly match the raw police tabular data.

In [ ]:
%%sql



DROP TABLE IF EXISTS sa2_crime_information;

CREATE TABLE sa2_crime_information AS
WITH spatial_join AS (
    SELECT
        sa2.sa2_code,
        sa2.sa2_name,
        c.total_crimes,
        ST_Area(ST_Intersection(sa2.geometry, sub.geometry)::geography) / 
        ST_Area(sub.geometry::geography) AS overlap_ratio

        FROM sa2_boundaries sa2

        JOIN suburbs sub ON ST_Intersects(sa2.geometry, sub.geometry)

        JOIN crime_raw c ON SPLIT_PART(sub.sal_name, ' (', 1) = c.suburb
   
)
SELECT 
    sa2_code,
    sa2_name,

    ROUND(SUM(total_crimes * overlap_ratio)::numeric, 0) AS crime_count
FROM spatial_join
GROUP BY sa2_code, sa2_name;

CREATE INDEX IF NOT EXISTS idx_sa2_crime ON sa2_crime_information(sa2_code);


Creates `sa2_information` to consolidate all the information needed to calculate the score of every single sa2 region so it is easy to access the data. Also creates indexes on `sa2_code` and `sa2_name` to improve query performance.

In [ ]:
%%sql

DROP TABLE IF EXISTS sa2_information;

CREATE TABLE sa2_information AS
SELECT 
    s.sa2_code, 
    s.sa2_name,
    s.areasqkm AS area_sqkm,
    COALESCE(b.business_count, 0) AS business_count, 
    COALESCE(p.total_people, 0) AS total_people,  
    COALESCE(st.stop_count, 0) AS stop_count,
    COALESCE(sc.school_count, 0) AS school_count, 
    COALESCE(y.young_population, 0) AS young_population,
    COALESCE(poi.poi_count, 0) AS poi_count,
    COALESCE(ci.crime_count, 0) AS crime_count
FROM sa2_boundaries s 
LEFT JOIN selected_businesses b ON s.sa2_code::bigint = b.sa2_code::bigint 
LEFT JOIN population p ON s.sa2_code::bigint = p.sa2_code::bigint 
LEFT JOIN sa2_stop_counts st ON s.sa2_code::bigint = st.sa2_code::bigint 
LEFT JOIN schools_per_sa2 sc ON s.sa2_code::bigint = sc.sa2_code::bigint 
LEFT JOIN young_population y ON s.sa2_code::bigint = y.sa2_code::bigint 
LEFT JOIN selected_pois poi ON s.sa2_code = poi.sa2_code
LEFT JOIN sa2_crime_information ci ON s.sa2_code::bigint = ci.sa2_code::bigint;
CREATE INDEX IF NOT EXISTS idx_sa2_code ON public.sa2_information (sa2_code);
CREATE INDEX IF NOT EXISTS idx_sa2_name ON public.sa2_information (sa2_name);


Creates `sa2_averages` to contain the average and standard deviation value for each factor needed to calculate the z-score. This table was created to avoid having to keep performing the same function repetitively. 

In [ ]:
%%sql


DROP TABLE IF EXISTS sa2_averages;

CREATE TABLE sa2_averages AS
SELECT
    -- Business (Per Capita)
    AVG((business_count::numeric / NULLIF(total_people, 0)) * 1000) AS business_avg,
    STDDEV((business_count::numeric / NULLIF(total_people, 0)) * 1000) AS business_stddev,
    
    -- Stops (Per Sq Km - Walkability)
    AVG(stop_count::numeric / NULLIF(area_sqkm, 0)) AS stop_density_avg,
    STDDEV(stop_count::numeric / NULLIF(area_sqkm, 0)) AS stop_density_stddev,
    
    -- Schools (Per Student - Capacity)
    AVG((school_count::numeric / NULLIF(young_population, 0)) * 1000) AS school_avg,
    STDDEV((school_count::numeric / NULLIF(young_population, 0)) * 1000) AS school_stddev,
    
    -- POIs (Per Sq Km - Access)
    AVG(poi_count::numeric / NULLIF(area_sqkm, 0)) AS poi_density_avg,
    STDDEV(poi_count::numeric / NULLIF(area_sqkm, 0)) AS poi_density_stddev,

    AVG((crime_count::numeric / NULLIF(total_people, 0)) * 1000) AS crime_avg,
    STDDEV((crime_count::numeric / NULLIF(total_people, 0)) * 1000) AS crime_stddev



    

FROM sa2_information
WHERE total_people > 500; 

Creates `z_scores` with the z scores for every single sa2 region and the final score for each region

In [ ]:
%%sql

DROP TABLE IF EXISTS z_scores;

CREATE TABLE z_scores AS
WITH base_scores AS (
    SELECT
        s.sa2_code,
        s.sa2_name,
        (((s.business_count::numeric / NULLIF(s.total_people, 0)) * 1000) - a.business_avg) / NULLIF(a.business_stddev, 0) AS business_z,
        ((s.stop_count::numeric / NULLIF(s.area_sqkm, 0)) - a.stop_density_avg) / NULLIF(a.stop_density_stddev, 0) AS stop_z,
        (((s.school_count::numeric / NULLIF(s.young_population, 0)) * 1000) - a.school_avg) / NULLIF(a.school_stddev, 0) AS school_z,
        ((s.poi_count::numeric / NULLIF(s.area_sqkm, 0)) - a.poi_density_avg) / NULLIF(a.poi_density_stddev, 0) AS poi_z,
        ((((s.crime_count::numeric / NULLIF(s.total_people, 0)) * 1000) - a.crime_avg) / NULLIF(a.crime_stddev, 0)) * -1 AS crime_z
    FROM sa2_information s, sa2_averages a
    WHERE s.total_people > 500
)
SELECT 
    sa2_code,
    sa2_name,
    business_z,
    stop_z,
    school_z,
    poi_z,
    crime_z,
    ROUND((COALESCE(business_z, 0) + COALESCE(stop_z, 0) + COALESCE(school_z, 0) + COALESCE(poi_z, 0) + COALESCE(crime_z, 0))::numeric, 4) AS total_z
FROM base_scores;

SELECT column_name 
FROM information_schema.columns 
WHERE table_name = 'z_scores';

Creates `sa2_scores_by_sa4` containing the SA2 codes, names, and scores for all SA2 regions within a specified SA4 region.

In [ ]:
%%sql

DROP TABLE IF EXISTS sa2_scores_by_sa4;

CREATE TABLE sa2_scores_by_sa4 AS
SELECT 
    z.sa2_code, 
    z.sa2_name,
    sb.sa4_name,
    z.total_z,
    -- 1. Global Rank: How this suburb compares to ALL of Sydney (0 to 1)
    PERCENT_RANK() OVER (ORDER BY z.total_z ASC) as global_rank_score,
    
    -- 2. Local Rank: How this suburb compares ONLY to its neighbors in the same SA4
    PERCENT_RANK() OVER (PARTITION BY sb.sa4_name ORDER BY z.total_z ASC) as local_rank_score
FROM public.z_scores z
JOIN public.sa2_boundaries sb
    ON z.sa2_code = sb.sa2_code;

SELECT * FROM sa2_scores_by_sa4 ORDER BY total_z DESC;


In [ ]:

query = """
SELECT 
    s.sa2_code,
    s.sa2_name,
    s.sa4_name,
    s.total_z,
    s.global_rank_score,
    s.local_rank_score,
    b.geometry
FROM sa2_scores_by_sa4 s
JOIN sa2_boundaries b ON s.sa2_code = b.sa2_code
"""


gdf_scores = gpd.read_postgis(query, engine, geom_col='geometry')

print(gdf_scores.head())

In [ ]:

fig = px.choropleth_map(
    gdf_scores,
    geojson=gdf_scores.geometry,
    locations=gdf_scores.index,
    color="global_rank_score",
    color_continuous_scale="Viridis",
    range_color=(0, 1),
    map_style="carto-positron",
    zoom=9,
    center = {"lat": -33.8688, "lon": 151.2093},
    opacity=0.5,
    hover_name="sa2_name",
    hover_data=["sa4_name", "total_z", "local_rank_score"], 
    labels={
        'global_rank_score':'Liveability Percentile', 
        'total_z': 'Raw Z-Score'
    }
)

fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.show()